# Busan Traffic AI PBL - 08_busan_weather_api.ipynb

부산시 교통량 예측 실습을 부산시 교통량 예측 프로젝트로 변경한 노트북입니다.
API 인증키와 ngrok token은 코드에 직접 넣지 말고 환경변수로 관리합니다.


In [ ]:
import requests
import json

import os
SERVICE_KEY = os.getenv("DATA_GO_KR_SERVICE_KEY", "")

url = "https://apis.data.go.kr/1360000/VilageFcstInfoService_2.0/getUltraSrtNcst"

params = {
    "serviceKey": SERVICE_KEY,
    "pageNo": 1,
    "numOfRows": 1000,
    "dataType": "JSON",
    "base_date": "20260119",
    "base_time": "1200",
    "nx": 98,
    "ny": 76
}

resp = requests.get(url, params=params, timeout=20)

print("status", resp.status_code)
print(resp.text[:500])

In [ ]:
import pandas as pd

data = resp.json()
items = data["response"]["body"]["items"]["item"]

df = pd.DataFrame(items)
df.head()

In [ ]:
code_map = {
    "T1H": "기온(°C)",
    "REH": "습도(%)",
    "RN1": "1시간 강수량(mm)",
    "PTY": "강수형태(코드)",
    "WSD": "풍속(m/s)",
    "VEC": "풍향(도)",
}

df["category_name"] = df["category"].map(code_map).fillna(df["category"])
df[["baseDate","baseTime","nx","ny","category","category_name","obsrValue"]].head(10)

In [ ]:
summary = df.pivot_table(index=["baseDate","baseTime","nx","ny"],
                         columns="category",
                         values="obsrValue",
                         aggfunc="first").reset_index()

summary

In [ ]:
summary[["baseDate","baseTime","T1H","REH","RN1","PTY"]]

In [ ]:
pty_map = {
    "0": "없음",
    "1": "비",
    "2": "비/눈",
    "3": "눈",
    "5": "빗방울",
    "6": "빗방울/눈날림",
    "7": "눈날림",
}

summary["PTY_desc"] = summary["PTY"].map(pty_map)
summary[["baseDate","baseTime","T1H","REH","RN1","PTY","PTY_desc"]]